#### Load the data


In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt


In [2]:
%pip install ucimlrepo


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from ucimlrepo import fetch_ucirepo

breast_cancer = fetch_ucirepo(id=17)


### Explore the Data


In [4]:
X = breast_cancer.data.features
y = breast_cancer.data.targets.squeeze()
X.shape, y.shape, X.info(), y.info(), y.value_counts()


<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavity2         

((569, 30),
 (569,),
 None,
 None,
 Diagnosis
 B    357
 M    212
 Name: count, dtype: int64)

#### Check for duplicates


In [5]:
X.duplicated().sum()


np.int64(0)

In [6]:
y = y.replace({"B": 0, "M": 1})
y = y.astype(int)


#### Split the dataset


In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, y_train.shape, X_test.shape, y_test.shape


((455, 30), (455,), (114, 30), (114,))

**Preprocessing**: The dataset contains no missing values, and all predictor variables are numeric, so imputation and one-hot encoding are not required. Standardization is applied within the pipelines for Logistic Regression and SVC. Tree-based models (Random Forest, Gradient Boosting, and XGBoost) are evaluated without feature scaling.


### Define models, apply transformation, and evaluate the models


**Define Candidate Model Pipelines**


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

GB_pipeline = Pipeline([("clf", GradientBoostingClassifier(random_state=42))])
XGB_pipeline = Pipeline([("clf", xgb.XGBClassifier(random_state=42))])
LR_pipeline = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())])
RF_pipeline = Pipeline([("clf", RandomForestClassifier(random_state=42))])
SVC_pipeline = Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="rbf", C=1.0, gamma="scale"))])


### Standardized Cross Validation Evaluation


In [9]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import time
import numpy as np
import pandas as pd
import json, joblib
from pathlib import Path

RANDOM_SEEDS = [11, 42, 67]
N_REPEATS = len(RANDOM_SEEDS)
CV_FOLDS = 5
PIPELINES = {"LR": LR_pipeline, "SVC": SVC_pipeline, "RF": RF_pipeline, "GB": GB_pipeline, "XGB": XGB_pipeline}


In [10]:
def evaluate_model(model_name, model, x, y, cv):
    fold_accuracies, fold_precisions, fold_recalls, fold_f1s, fold_aucs = [], [], [], [], []
    y_true_all, y_pred_all = [], []
    start_time = time.perf_counter()
    fit_count = 0
    for train_index, val_index in cv.split(X=x, y=y):
        clf = clone(model)
        X_train_fold, y_train_fold = x.iloc[train_index], y.iloc[train_index]
        X_val_fold, y_val_fold = x.iloc[val_index], y.iloc[val_index]
        clf.fit(X_train_fold, y_train_fold)
        fit_count += 1
        y_pred = clf.predict(X_val_fold)
        if hasattr(clf, "predict_proba"):
            y_score = clf.predict_proba(X_val_fold)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_score = clf.decision_function(X_val_fold)
        else:
            raise ValueError(f"{model_name} cannot produce scores for AUC")
        y_true_all.extend(y_val_fold)
        y_pred_all.extend(y_pred)
        fold_accuracies.append(accuracy_score(y_val_fold, y_pred))
        fold_precisions.append(precision_score(y_val_fold, y_pred, zero_division=0))
        fold_recalls.append(recall_score(y_val_fold, y_pred, zero_division=0))
        fold_f1s.append(f1_score(y_val_fold, y_pred, zero_division=0))
        fold_aucs.append(roc_auc_score(y_val_fold, y_score))
    runtime = time.perf_counter() - start_time
    cm = confusion_matrix(y_true_all, y_pred_all)
    return {"Model": model_name, "Accuracy": np.mean(fold_accuracies), "Accuracy_std": np.std(fold_accuracies, ddof=1),
        "Precision": np.mean(fold_precisions), "Precision_std": np.std(fold_precisions, ddof=1),
        "Recall": np.mean(fold_recalls), "Recall_std": np.std(fold_recalls, ddof=1), "F1": np.mean(fold_f1s),
        "F1_std": np.std(fold_f1s, ddof=1), "AUC": np.mean(fold_aucs), "AUC_std": np.std(fold_aucs, ddof=1),
        "Runtime_seconds": runtime, "Model_evaluations": 1, "CV_fits": fit_count, "Confusion_matrix": cm}


In [11]:
repeated_results = []
for repeat, seed in enumerate(RANDOM_SEEDS, start=1):
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    for model_name, model in PIPELINES.items():
        result = evaluate_model(model_name, model, X_train, y_train, cv)
        result.update({"Repeat": repeat, "Seed": seed, "CV_Folds": CV_FOLDS,
            "Development_CV_F1_Mean": result["F1"], "Development_CV_F1_SD": result["F1_std"],
            "Development_CV_Runtime_Seconds": result["Runtime_seconds"]})
        repeated_results.append(result)
repeated_results_df = pd.DataFrame(repeated_results)


In [12]:
# Summarize each CV metric across the three repeated runs for every pipeline.
metric_names = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
summary_aggregations = {}
for metric in metric_names:
    summary_aggregations[f"Development_CV_{metric}_Mean"] = (metric, "mean")
    summary_aggregations[f"Development_CV_{metric}_SD"] = (metric, "std")
summary_aggregations["Development_CV_Runtime_Seconds_Mean"] = ("Development_CV_Runtime_Seconds", "mean")
summary_aggregations["Development_CV_Runtime_Seconds_SD"] = ("Development_CV_Runtime_Seconds", "std")
summary_df = (repeated_results_df.groupby("Model", as_index=False)
    .agg(**summary_aggregations)
    .sort_values("Development_CV_F1_Mean", ascending=False).reset_index(drop=True))
display(summary_df.round(4))


,Model,Development_CV_Accuracy_Mean,Development_CV_Accuracy_SD,Development_CV_Precision_Mean,Development_CV_Precision_SD,Development_CV_Recall_Mean,Development_CV_Recall_SD,Development_CV_F1_Mean,Development_CV_F1_SD,Development_CV_AUC_Mean,Development_CV_AUC_SD,Development_CV_Runtime_Seconds_Mean,Development_CV_Runtime_Seconds_SD
0,SVC,0.9714,0.0000,0.9712,0.0062,0.9529,0.0059,0.9614,0.0004,0.9940,0.0013,0.2204,0.0485
1,LR,0.9714,0.0022,0.9746,0.0040,0.9490,0.0068,0.9610,0.0031,0.9946,0.0012,0.2329,0.0312
2,XGB,0.9656,0.0051,0.9619,0.0151,0.9471,0.0118,0.9536,0.0066,0.9907,0.0016,2.2032,0.1731
3,GB,0.9590,0.0077,0.9521,0.0122,0.9392,0.0090,0.9445,0.0109,0.9893,0.0026,2.9410,0.0534
4,RF,0.9575,0.0063,0.9530,0.0096,0.9353,0.0059,0.9429,0.0078,0.9875,0.0029,1.7859,0.0960


### Final Development Fit and Saved Model


In [13]:
best_model_name = summary_df.iloc[0]["Model"]
best_pipeline = clone(PIPELINES[best_model_name])
workflow_cpu_start = time.process_time()
fit_start, fit_cpu_start = time.perf_counter(), time.process_time()
best_pipeline.fit(X_train, y_train)
final_development_fit_seconds = time.perf_counter() - fit_start
final_fit_cpu_seconds = time.process_time() - fit_cpu_start
folder_name = "eval - all features"
output_dir = Path.cwd() if Path.cwd().name == folder_name else Path.cwd() / folder_name
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "conventional_frozen_model.joblib"
save_start = time.perf_counter(); joblib.dump(best_pipeline, model_path); model_save_seconds = time.perf_counter() - save_start
load_start = time.perf_counter(); loaded_best_pipeline = joblib.load(model_path); model_load_seconds = time.perf_counter() - load_start
final_workflow_wall_seconds = final_development_fit_seconds + model_save_seconds + model_load_seconds
best_record = {"Best_Model": best_model_name, "CV_Folds": CV_FOLDS, "Random_Seeds": RANDOM_SEEDS,
    "Development_CV_F1_Mean": float(summary_df.iloc[0]["Development_CV_F1_Mean"]),
    "Development_CV_F1_SD": float(summary_df.iloc[0]["Development_CV_F1_SD"]),
    "Development_CV_Runtime_Seconds": float(summary_df.iloc[0]["Development_CV_Runtime_Seconds_Mean"]),
    "Agent_Runtime_Seconds": None, "Final_Development_Fit_Runtime_Seconds": final_development_fit_seconds,
    "Model_Save_Runtime_Seconds": model_save_seconds, "Model_Load_Runtime_Seconds": model_load_seconds,
    "Final_Test_Scoring_Wall_Seconds": None, "Final_Test_Scoring_CPU_Seconds": None,
    "Final_Workflow_Wall_Seconds": final_workflow_wall_seconds, "Final_Workflow_CPU_Seconds": time.process_time() - workflow_cpu_start,
    "Test_Scored": False, "Saved_Model_Path": str(model_path)}
for metric in ["Accuracy", "Precision", "Recall", "F1", "AUC"]:
    best_record[f"Development_CV_{metric}_Mean"] = float(summary_df.iloc[0][f"Development_CV_{metric}_Mean"])
    best_record[f"Development_CV_{metric}_SD"] = float(summary_df.iloc[0][f"Development_CV_{metric}_SD"])
best_record["Development_CV_Runtime_Seconds_Mean"] = float(summary_df.iloc[0]["Development_CV_Runtime_Seconds_Mean"])
best_record["Development_CV_Runtime_Seconds_SD"] = float(summary_df.iloc[0]["Development_CV_Runtime_Seconds_SD"])
# Write reproducibility artifacts matching the useful outputs from the agentic workflow.
for seed in RANDOM_SEEDS:
    repeated_results_df.loc[repeated_results_df["Seed"] == seed].to_csv(
        output_dir / f"conventional_seed_{seed}_runs.csv", index=False)

seed_winners = (repeated_results_df.sort_values("Development_CV_F1_Mean", ascending=False)
    .groupby("Seed", as_index=False).first()[["Seed", "Model", "Development_CV_F1_Mean"]]
    .rename(columns={"Model": "Selected_Model", "Development_CV_F1_Mean": "Selected_Model_CV_F1"}))
seed_winners.to_csv(output_dir / "conventional_seed_selections.csv", index=False)
decision_consistency = (seed_winners.groupby("Selected_Model", as_index=False)
    .agg(Seed_Count=("Seed", "count"), Seeds=("Seed", lambda values: list(values)))
    .sort_values("Seed_Count", ascending=False))
decision_consistency.to_csv(output_dir / "decision_consistency.csv", index=False)

split_rows = pd.concat([
    pd.DataFrame({"Dataset_Row_Index": X_train.index.to_numpy(), "Partition": "development", "Target": y_train.to_numpy()}),
    pd.DataFrame({"Dataset_Row_Index": X_test.index.to_numpy(), "Partition": "test", "Target": y_test.to_numpy()})
], ignore_index=True)
split_rows.to_csv(output_dir / "split_manifest.csv", index=False)
partition_manifest = {
    "Split_Random_State": 42,
    "Test_Size_Fraction": 0.2,
    "Stratified": True,
    "Development_Row_Count": int(len(X_train)),
    "Test_Row_Count": int(len(X_test)),
    "Development_Target_Counts": {str(k): int(v) for k, v in y_train.value_counts().sort_index().items()},
    "Test_Target_Counts": {str(k): int(v) for k, v in y_test.value_counts().sort_index().items()}
}
(output_dir / "data_partition_manifest.json").write_text(json.dumps(partition_manifest, indent=2), encoding="utf-8")
run_metadata = {
    "Workflow": "conventional_pipeline_evaluation",
    "Random_Seeds": RANDOM_SEEDS,
    "CV_Folds": CV_FOLDS,
    "Repeated_Runs_Per_Pipeline": len(RANDOM_SEEDS),
    "Split_Random_State": 42,
    "Selection_Metric": "mean development CV F1 across seeds",
    "Test_Scored": False
}
(output_dir / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
summary_df.to_csv(output_dir / "all_pipelines_summary.csv", index=False)
summary_df.to_csv(output_dir / "conventional_summary.csv", index=False)
pd.DataFrame([best_record]).to_csv(output_dir / "conventional_final_run.csv", index=False)
(output_dir / "conventional_final_run.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
(output_dir / "conventional_frozen_workflow.json").write_text(json.dumps({
    "Workflow_State": "development_fit_complete_test_pending",
    "Selected_Model": best_model_name,
    "Saved_Model_Path": str(model_path),
    "Selection_Rule": run_metadata["Selection_Metric"],
    "Random_Seeds": RANDOM_SEEDS,
    "Split_Manifest": str(output_dir / "split_manifest.csv"),
    "Run_Metadata": str(output_dir / "run_metadata.json")
}, indent=2), encoding="utf-8")

repeated_results_df.to_csv(output_dir / "conventional_repeated_runs.csv", index=False)
summary_df.to_csv(output_dir / "conventional_pipeline_summary.csv", index=False)
(output_dir / "best_pipeline_record.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
print("Selected pipeline:", best_model_name)
print("Saved development-only model and record:", model_path)
print("Test set was not scored.")


Selected pipeline: SVC
Saved development-only model and record: c:\Users\ogeuc\Downloads\Conventional\eval - all features\conventional_frozen_model.joblib
Test set was not scored.


### Final Test Set Evaluation

This cell loads the saved best pipeline and evaluates it on the held-out test set. 

In [ ]:
# Run this cell only when ready to evaluate the held-out test set.
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

loaded_best_pipeline = joblib.load(model_path)
test_wall_start = time.perf_counter()
test_cpu_start = time.process_time()
y_test_pred = loaded_best_pipeline.predict(X_test)
if hasattr(loaded_best_pipeline, "predict_proba"):
    class_index = list(loaded_best_pipeline.classes_).index(1)
    y_test_score = loaded_best_pipeline.predict_proba(X_test)[:, class_index]
else:
    y_test_score = loaded_best_pipeline.decision_function(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_auc = roc_auc_score(y_test, y_test_score)
test_scoring_cpu_seconds = time.process_time() - test_cpu_start
test_scoring_seconds = time.perf_counter() - test_wall_start

best_record.update({
    "Final_Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Final_Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds,
    "Test_Scored": True,
    "Final_Test_Accuracy": test_accuracy,
    "Final_Test_Precision": test_precision,
    "Final_Test_Recall": test_recall,
    "Final_Test_F1": test_f1,
    "Final_Test_ROC_AUC": test_auc,
    "Final_Test_Confusion_Matrix": confusion_matrix(y_test, y_test_pred).tolist()
})
test_metrics = {
    "Model": best_model_name,
    "Test_Accuracy": test_accuracy,
    "Test_Precision": test_precision,
    "Test_Recall": test_recall,
    "Test_F1": test_f1,
    "Test_ROC_AUC": test_auc,
    "Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds
}
pd.DataFrame([test_metrics]).to_csv(output_dir / "conventional_final_test.csv", index=False)
(output_dir / "conventional_frozen_workflow.json").write_text(json.dumps({
    "Workflow_State": "test_scoring_complete",
    "Selected_Model": best_model_name,
    "Saved_Model_Path": str(model_path),
    "Test_Metrics_Path": str(output_dir / "conventional_final_test.csv"),
    "Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds,
    "Random_Seeds": RANDOM_SEEDS
}, indent=2), encoding="utf-8")
(output_dir / "conventional_final_run.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
pd.DataFrame([best_record]).to_csv(output_dir / "conventional_final_run.csv", index=False)

(output_dir / "best_pipeline_record.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
print("Selected pipeline:", best_model_name)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test precision: {test_precision:.4f}")
print(f"Test recall: {test_recall:.4f}")
print(f"Test F1: {test_f1:.4f}")
print(f"Test ROC AUC: {test_auc:.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification report:\n", classification_report(y_test, y_test_pred, zero_division=0))
